# 🎬 Movie Recommender

**Goal:** recommend movies a person will like — the same idea behind Netflix, YouTube, Spotify and Amazon.

**New skills in this project**
- **Collaborative filtering** — "people who liked X also liked Y" — without knowing anything about the movies themselves
- Working with a huge, mostly empty **user × movie matrix**
- **Cosine similarity**
- **Embeddings** — learning a small vector of numbers that captures a movie's "taste profile"
- **Matrix factorisation** in PyTorch
- Evaluating recommenders: **RMSE** and **hit rate @ 10**
- Recommending for a **brand-new user** (you!)

⏱️ One to two weeks. **How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

⚡ **Turn on the free GPU first:** in Colab, **Runtime → Change runtime type → T4 GPU → Save**. Training will be 10–50× faster.

## 1. Load MovieLens
MovieLens (from the University of Minnesota) — we'll use the *small* version: ~100,000 ratings (0.5–5 stars) by 610 users on ~9,700 movies.

In [ ]:
import os, zipfile, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists("ml-latest-small.zip"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml-latest-small.zip")
with zipfile.ZipFile("ml-latest-small.zip") as z:
    z.extractall(".")

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")
print(ratings.shape, movies.shape)
ratings.head()

In [ ]:
movies.head()

## 2. Explore

In [ ]:
n_users, n_movies = ratings["userId"].nunique(), ratings["movieId"].nunique()
density = len(ratings) / (n_users * n_movies)
print(f"{n_users} users × {n_movies} movies = {n_users * n_movies:,} possible ratings")
print(f"We have {len(ratings):,} → the matrix is {density:.1%} full ({1 - density:.1%} empty!)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ratings["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], title="How people rate")
ratings.groupby("movieId").size().sort_values(ascending=False).reset_index(drop=True).plot(
    ax=axes[1], logy=True, title="Ratings per movie (sorted) — the 'long tail'")
axes[1].set_xlabel("movie rank"); plt.tight_layout(); plt.show()

👀 Two big challenges for recommenders:
1. **Sparsity** — each user has rated a tiny fraction of movies. Our job is to fill in the blanks.
2. **The long tail** — a few blockbusters get most ratings; most movies have very few.

In [ ]:
stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).join(movies.set_index("movieId")["title"])
print("Most-rated movies:")
print(stats.sort_values("count", ascending=False).head(10)[["title", "count", "mean"]].round(2).to_string())

In [ ]:
def find_movie(text):
    """Search movies by (part of) the title."""
    hits = stats[stats["title"].str.contains(text, case=False, regex=False)]
    return hits.sort_values("count", ascending=False)[["title", "count", "mean"]].round(2)

find_movie("Matrix")

## 3. Train/test split

For each user we hide **20% of their ratings** as a test set. The model must predict those hidden ratings.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42, stratify=ratings["userId"])
test = test[test["movieId"].isin(train["movieId"])]      # can't predict movies nobody rated in training
print(len(train), "train ratings /", len(test), "test ratings")

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

## 4. Baselines

1. **Global average** — predict the same rating for everything.
2. **Bias model** — some users are generous, some movies are just better:
   `prediction = global average + user bias + movie bias`

In [ ]:
mu = train["rating"].mean()
print(f"Global average ({mu:.2f}) RMSE: {rmse(test['rating'], np.full(len(test), mu)):.3f}")

# Movie bias (shrunk towards 0 for movies with few ratings — "regularisation")
reg = 5
m_stats = train.groupby("movieId")["rating"].agg(["sum", "count"])
movie_bias = (m_stats["sum"] - mu * m_stats["count"]) / (m_stats["count"] + reg)
t = train.assign(resid=train["rating"] - mu - train["movieId"].map(movie_bias))
u_stats = t.groupby("userId")["resid"].agg(["sum", "count"])
user_bias = u_stats["sum"] / (u_stats["count"] + reg)

bias_pred = mu + test["userId"].map(user_bias).fillna(0) + test["movieId"].map(movie_bias).fillna(0)
print(f"Bias model RMSE: {rmse(test['rating'], bias_pred.clip(0.5, 5)):.3f}")

RMSE is in stars: the bias model is off by roughly **0.87 stars** on average. Now let's make it *personal*.

## 5. Item-based collaborative filtering — "people who liked X also liked Y"

Put ratings into a big **user × movie matrix**. Each movie becomes a column of ratings. Two movies are **similar** if the *same people* rated them in the same way.

We measure similarity with **cosine similarity** (the angle between two vectors: 1 = same direction, 0 = unrelated). We first subtract each user's average so generous and harsh raters are comparable.

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

user_ids = np.sort(train["userId"].unique())
movie_ids = np.sort(train["movieId"].unique())
u_index = {u: i for i, u in enumerate(user_ids)}
m_index = {m: i for i, m in enumerate(movie_ids)}

centered = train["rating"] - train.groupby("userId")["rating"].transform("mean")
R = csr_matrix((centered, (train["userId"].map(u_index), train["movieId"].map(m_index))),
               shape=(len(user_ids), len(movie_ids)))
print("rating matrix:", R.shape)

popular = m_stats["count"].reindex(movie_ids).fillna(0).values >= 20     # only compare well-known movies
item_sim = cosine_similarity(R.T)           # movie × movie similarity
print("similarity matrix:", item_sim.shape)

In [ ]:
def similar_movies(title_part, n=10):
    mid = find_movie(title_part).index[0]
    i = m_index[mid]
    sims = pd.Series(item_sim[i], index=movie_ids)[popular]
    sims = sims.drop(mid, errors="ignore").sort_values(ascending=False).head(n)
    print(f"Because you liked {stats.loc[mid, 'title']}:")
    return pd.DataFrame({"title": stats.loc[sims.index, "title"].values, "similarity": sims.values.round(3)})

similar_movies("Toy Story (1995)")

In [ ]:
similar_movies("Matrix, The")

It has never seen a genre label or plot summary — it learned this purely from **who rated what**. That's collaborative filtering.

## 6. Embeddings and matrix factorisation

Imagine every movie could be described by a few hidden "taste" numbers — how much action, romance, how artsy, how kid-friendly… — and every user by how much they *like* each of those. Then:

> predicted rating = (user's taste vector) · (movie's vector) + biases

We **don't** tell the model what the numbers mean. It **learns** them from ratings. These learned vectors are called **embeddings** — the same idea powers word embeddings, image search and LLMs.

```
         movies                         k factors         movies
users [ ? 4 ? 5 ... ]   ≈   users [ user vectors ] × k [ movie vectors ]
```

In [ ]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, k=32):
        super().__init__()
        self.user_vec = nn.Embedding(n_users, k)
        self.movie_vec = nn.Embedding(n_movies, k)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        for e in [self.user_vec, self.movie_vec]:
            nn.init.normal_(e.weight, std=0.05)
        for e in [self.user_bias, self.movie_bias]:
            nn.init.zeros_(e.weight)
        self.mu = nn.Parameter(torch.tensor(float(mu)), requires_grad=False)

    def forward(self, u, m):
        dot = (self.user_vec(u) * self.movie_vec(m)).sum(1)
        return self.mu + dot + self.user_bias(u).squeeze(1) + self.movie_bias(m).squeeze(1)

def to_tensors(df):
    u = torch.tensor(df["userId"].map(u_index).values, dtype=torch.long)
    m = torch.tensor(df["movieId"].map(m_index).values, dtype=torch.long)
    r = torch.tensor(df["rating"].values, dtype=torch.float32)
    return u.to(device), m.to(device), r.to(device)

tr_u, tr_m, tr_r = to_tensors(train)
te_u, te_m, te_r = to_tensors(test)
mf = MatrixFactorization(len(user_ids), len(movie_ids), k=32).to(device)
print(mf)

Training uses the same loop as the digit recognizer, plus a **regularisation** penalty (`LAMBDA`) that keeps the embedding numbers small. Without it, the model memorises the training ratings (overfitting). Try `LAMBDA = 0` later and watch the test RMSE get *worse* every epoch.

In [ ]:
opt = torch.optim.Adam(mf.parameters(), lr=0.005)
BATCH, EPOCHS, LAMBDA = 1024, 20, 0.1
history = []
for epoch in range(1, EPOCHS + 1):
    mf.train()
    perm = torch.randperm(len(tr_r), device=device)
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i + BATCH]
        pred = mf(tr_u[idx], tr_m[idx])
        u, m = tr_u[idx], tr_m[idx]
        penalty = ((mf.user_vec(u) ** 2).sum(1) + (mf.movie_vec(m) ** 2).sum(1)
                   + mf.user_bias(u).squeeze(1) ** 2 + mf.movie_bias(m).squeeze(1) ** 2).mean()
        loss = ((pred - tr_r[idx]) ** 2).mean() + LAMBDA * penalty
        opt.zero_grad(); loss.backward(); opt.step()
    mf.eval()
    with torch.no_grad():
        tr_rmse = rmse(tr_r.cpu(), mf(tr_u, tr_m).clamp(0.5, 5).cpu())
        te_rmse = rmse(te_r.cpu(), mf(te_u, te_m).clamp(0.5, 5).cpu())
    history.append((tr_rmse, te_rmse))
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}: train RMSE {tr_rmse:.3f} | test RMSE {te_rmse:.3f}")

In [ ]:
h = np.array(history)
plt.figure(figsize=(8, 4))
plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="test")
plt.xlabel("epoch"); plt.ylabel("RMSE (stars)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Matrix factorisation training"); plt.show()
best_epoch = int(h[:, 1].argmin()) + 1
print(f"Best test RMSE {h[:, 1].min():.3f} at epoch {best_epoch}")

👀 If the test curve starts rising while training keeps falling — that's **overfitting**. With `LAMBDA = 0.1` both curves settle; try `0`, `0.02` and `0.3` to see the trade-off (re-run from the cell that creates `mf`).

## 7. Exploring the learned embeddings
Movies with similar embedding vectors should feel similar — even though we never told the model anything about them.

In [ ]:
movie_emb = mf.movie_vec.weight.detach().cpu().numpy()
emb_sim = cosine_similarity(movie_emb)

def similar_by_embedding(title_part, n=10):
    mid = find_movie(title_part).index[0]
    sims = pd.Series(emb_sim[m_index[mid]], index=movie_ids)[popular].drop(mid, errors="ignore")
    top = sims.sort_values(ascending=False).head(n)
    print(f"Nearest to {stats.loc[mid, 'title']} in embedding space:")
    return pd.DataFrame({"title": stats.loc[top.index, "title"].values, "similarity": top.values.round(3)})

similar_by_embedding("Toy Story (1995)")

In [ ]:
from sklearn.decomposition import PCA

top_ids = stats.sort_values("count", ascending=False).head(40).index
pts = PCA(n_components=2).fit_transform(movie_emb[[m_index[m] for m in top_ids]])
plt.figure(figsize=(13, 9))
plt.scatter(pts[:, 0], pts[:, 1], alpha=0.5)
for (x, y), mid in zip(pts, top_ids):
    plt.annotate(stats.loc[mid, "title"][:28], (x, y), fontsize=8)
plt.title("40 most popular movies — embeddings squashed to 2-D (similar movies land near each other)")
plt.show()

## 8. Recommending: hit rate @ 10

RMSE measures rating accuracy, but users only ever see a **top-10 list**. A more realistic test: for each user, take a movie they rated **highly (4.5+)** in the test set, mix it with 99 random movies they haven't rated, and check if our model ranks it in the **top 10**.

In [ ]:
rng = np.random.default_rng(0)
rated = train.groupby("userId")["movieId"].apply(set)
all_movies = np.array(movie_ids)
liked = test[test["rating"] >= 4.5].groupby("userId").head(1)

def hit_rate(score_fn, n_users=300):
    hits = 0
    sample = liked.sample(min(n_users, len(liked)), random_state=0)
    for _, row in sample.iterrows():
        u, target = row["userId"], row["movieId"]
        seen = rated.get(u, set()) | set(test.loc[test["userId"] == u, "movieId"])
        pool = rng.choice([m for m in all_movies if m not in seen], size=99, replace=False)
        candidates = np.r_[target, pool]
        scores = score_fn(u, candidates)
        hits += target in candidates[np.argsort(-scores)[:10]]
    return hits / len(sample)

popularity = m_stats["count"]
def popular_scores(u, cands):
    return popularity.reindex(cands).fillna(0).values

def mf_scores(u, cands):
    with torch.no_grad():
        uu = torch.full((len(cands),), u_index[u], dtype=torch.long, device=device)
        mm = torch.tensor([m_index[m] for m in cands], dtype=torch.long, device=device)
        return mf(uu, mm).cpu().numpy()

print(f"Random guess      hit@10: {10 / 100:.1%}")
print(f"Most popular      hit@10: {hit_rate(popular_scores):.1%}")
print(f"Matrix factorisat hit@10: {hit_rate(mf_scores):.1%}")

🤔 **Popularity wins?!** This is a famous lesson in recommender systems. Our model was trained to predict *how many stars* you'd give a movie **if you watched it** — not *which movies you're likely to watch*. The 99 random candidates are mostly obscure films nobody watches, and a user's "liked" movie is usually a well-known one. So "recommend what's popular" is a very strong baseline — **always check against it**.

Real systems **combine** signals. Let's blend the personal MF score with popularity and see if we get the best of both:

In [ ]:
log_pop = np.log1p(popularity)

def blended_scores(u, cands, w):
    mf_s = mf_scores(u, cands)
    pop_s = log_pop.reindex(cands).fillna(0).values
    z = lambda v: (v - v.mean()) / (v.std() + 1e-9)       # put both on the same scale
    return (1 - w) * z(mf_s) + w * z(pop_s)

for w in [0.3, 0.5, 0.7]:
    print(f"Blend (popularity weight {w}) hit@10: {hit_rate(lambda u, c: blended_scores(u, c, w)):.1%}")

With the right weight, the blend beats both: popularity decides *what people actually watch*, and the embeddings decide *which of those suit this particular person*.

## 9. 🍿 Recommendations for *you*

Rate some movies below (0.5–5). We keep the movie embeddings **frozen** and learn just *your* taste vector from your ratings — this is how services handle new users ("cold start").

In [ ]:
my_ratings = {                       # ✏️ edit these — use find_movie("...") to look up exact titles
    "Toy Story (1995)": 4.5,
    "Matrix, The (1999)": 5.0,
    "Titanic (1997)": 2.0,
    "Lord of the Rings: The Fellowship of the Ring, The (2001)": 5.0,
    "Notebook, The (2004)": 1.5,
    "Inception (2010)": 5.0,
}

title_to_id = stats.reset_index().set_index("title")["movieId"]
mine = {title_to_id[t]: r for t, r in my_ratings.items() if t in title_to_id.index and title_to_id[t] in m_index}
print(f"Using {len(mine)} of your ratings")

m_idx = torch.tensor([m_index[m] for m in mine], dtype=torch.long, device=device)
r_vals = torch.tensor(list(mine.values()), dtype=torch.float32, device=device)
my_vec = torch.zeros(mf.user_vec.weight.shape[1], device=device, requires_grad=True)
my_bias = torch.zeros(1, device=device, requires_grad=True)
opt_me = torch.optim.Adam([my_vec, my_bias], lr=0.05)
for _ in range(300):
    with torch.no_grad():
        mv, mb = mf.movie_vec(m_idx), mf.movie_bias(m_idx).squeeze(1)
    pred = mu + (mv * my_vec).sum(1) + mb + my_bias
    loss = ((pred - r_vals) ** 2).mean() + 0.05 * (my_vec ** 2).sum()
    opt_me.zero_grad(); loss.backward(); opt_me.step()

with torch.no_grad():
    all_scores = (mu + mf.movie_vec.weight @ my_vec + mf.movie_bias.weight.squeeze(1) + my_bias).cpu().numpy()
recs = pd.Series(all_scores, index=movie_ids)[popular].drop(list(mine), errors="ignore")
recs = recs.sort_values(ascending=False).head(15)
pd.DataFrame({"title": stats.loc[recs.index, "title"].values, "predicted stars": recs.values.clip(0.5, 5).round(2)})

In [ ]:
torch.save({"model": mf.state_dict(), "movie_ids": movie_ids, "user_ids": user_ids}, "movie_mf.pt")
print("Saved movie_mf.pt")

## ✅ What you learned
- **Collaborative filtering** recommends from behaviour alone.
- **Cosine similarity** finds similar items in a sparse user × item matrix.
- **Embeddings** learned by **matrix factorisation** capture hidden "taste" dimensions.
- Evaluate recommenders like users experience them (**hit rate @ 10**) and always compare with **popularity**.
- New users can be handled by learning just their vector (**cold start**).

## ✍️ Try it yourself
1. Change `k` (the embedding size) to 8 and 128. What happens to test RMSE?
2. **Content-based** recommendations: turn each movie's `genres` into a TF-IDF vector (like the spam project!) and find similar movies by genre. Then mix it with collaborative scores — a **hybrid** recommender.
3. Scale up to [MovieLens 1M or 25M](https://grouplens.org/datasets/movielens/). What changes?
4. Build a Gradio app: pick 5 movies you like from dropdowns → get 10 recommendations.

In [ ]:
# Your code here